# 01 | Dimensions: stores, products, dates and promotions

Creates the four dimension tables of a **fictitious** chain of sports and outdoor stores, *Cume & Costa*, with stores in Portugal and Spain.

> **All the data in this project is synthetic.** The company, stores, products, prices and sales are invented. Only the geography (postal codes, localities and NUTS regions) is real public data, from the project [postal-admin-reference-pt-es](../../postal-admin-reference-pt-es).

| Table | Content |
|---|---|
| `dim_store` | Stores, located at real postal codes, chosen in proportion to the size of the locality |
| `dim_product` | Product catalogue in six categories |
| `dim_date` | Calendar with weekdays, seasons and public holidays of Portugal and Spain |
| `dim_promotion` | Commercial calendar (sales, Black Friday, seasonal campaigns) |

The parameters (period, number of stores, brand, random seed) are in `src/config.py`.

## Setup

In [1]:
import sys
from pathlib import Path

# Make the src/ folder importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import *
from src.utils import *
import holidays

rng = np.random.default_rng(SEED)

## Reference data (real geography)

The CSV files exported by notebook 07 of the reference project. By default they are read from `../postal-admin-reference-pt-es/outputs/csv/` (both repositories in the same folder). Change `REFERENCE_CSV_DIR` in `src/config.py` if needed.

In [2]:
if not REFERENCE_CSV_DIR.exists():
    raise FileNotFoundError(
        f"Reference CSV folder not found: {REFERENCE_CSV_DIR}\n"
        "Run notebook 07 of postal-admin-reference-pt-es or change REFERENCE_CSV_DIR in src/config.py."
    )

ref_postal_codes = pd.read_csv(REFERENCE_CSV_DIR / "postal_codes_PT_ES.csv", dtype={"PostalCode": str})
ref_localities = pd.read_csv(REFERENCE_CSV_DIR / "localities_PT_ES.csv")
ref_nuts3 = pd.read_csv(REFERENCE_CSV_DIR / "nuts3_PT_ES.csv")

print(f"Postal codes: {len(ref_postal_codes):,} | Localities: {len(ref_localities):,} | NUTS 3: {len(ref_nuts3):,}")

Postal codes: 209,829 | Localities: 11,224 | NUTS 3: 85


### Check the Eurostat typology codes

The stores are classified as *Mountain*, *Coastal* or *Inland* using the NUTS 3 typologies. Confirm that the codes match the values in `src/config.py` (`MOUNTAIN_CODES`, `COASTAL_CODES`):
* **MountainType:** 1, 2, 3 = mountain regions; 4 = non-mountain
* **CoastalType:** 1, 2 = coastal regions; 3 = non-coastal
* **DegreeUrb** (postal codes): 1 = cities; 2 = towns and suburbs; 3 = rural areas

In [3]:
print(ref_nuts3["MountainType"].value_counts(dropna=False).sort_index(), "\n")
print(ref_nuts3["CoastalType"].value_counts(dropna=False).sort_index(), "\n")
print(ref_postal_codes["DegreeUrb"].value_counts(dropna=False).sort_index())

MountainType
1.0    46
NaN    39
Name: count, dtype: int64 

CoastalType
1.0    47
NaN    38
Name: count, dtype: int64 

DegreeUrb
1.0    73818
2.0    64305
3.0    71698
NaN        8
Name: count, dtype: int64


## Candidate locations

Each locality receives the NUTS 3 region of most of its postal codes, and with it the mountain and coastal classification.

**Size of each locality.** The Eurostat LAU file has the population of the Portuguese parishes, but **not of the Spanish municipalities** (their values are zero in the source, turned into missing values in the reference project). For a country without population, the **number of postal codes** of each locality is used instead: larger localities have more postal codes. The check below shows how many localities have population in each country.

In [4]:
# How many localities have population, by country
ref_localities.assign(Country=ref_localities["LocalityID"].str[:2]).groupby("Country")["Population2024"].agg(
    localities="size", with_population="count"
)

,localities,with_population
Country,,
ES,8132,0
PT,3092,3092


In [5]:
# NUTS 3 of each locality (the most frequent one among its postal codes)
locality_nuts3 = (
    ref_postal_codes.groupby("LocalityID")["NUTS3_ID"]
    .agg(lambda values: values.mode().iloc[0])
    .reset_index()
)

postal_codes_per_locality = ref_postal_codes.groupby("LocalityID").size().rename("PostalCodeCount").reset_index()

candidates = (
    ref_localities[["LocalityID", "LocalityName", "Population2024"]]
    .merge(locality_nuts3, on="LocalityID")
    .merge(postal_codes_per_locality, on="LocalityID")
    .merge(ref_nuts3[["NUTS3_ID", "NUTS3_Name", "MountainType", "CoastalType"]], on="NUTS3_ID")
)
candidates["Country"] = candidates["LocalityID"].str[:2]

# Size of each locality: the population when the country has it; otherwise, the number of postal codes
# (the Eurostat LAU file has no population for the Spanish municipalities - see the check above)
country_uses_population = candidates.groupby("Country")["Population2024"].transform(lambda pop: pop.notna().mean() >= 0.5)
candidates["SizeIndicator"] = np.where(country_uses_population, candidates["Population2024"], candidates["PostalCodeCount"])
candidates = candidates.dropna(subset=["SizeIndicator"])

print("Size indicator used:")
print(country_uses_population.groupby(candidates["Country"]).first()
      .map({True: "population", False: "number of postal codes"}).rename("size_indicator"))

candidates["IsMountain"] = candidates["MountainType"].isin(MOUNTAIN_CODES)
candidates["IsCoastal"] = candidates["CoastalType"].isin(COASTAL_CODES)

# Size tier (1 to 5) inside each country: localities are parishes in Portugal and
# municipalities in Spain, and the size indicator is not the same, so they are only comparable inside the same country
candidates["PopulationTier"] = (
    candidates.groupby("Country")["SizeIndicator"]
    .transform(lambda size: pd.qcut(size.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]))
    .astype(int)
)

candidates.groupby("Country")[["IsMountain", "IsCoastal"]].sum()

Size indicator used:
Country
ES    number of postal codes
PT                population
Name: size_indicator, dtype: str


,IsMountain,IsCoastal
Country,,
ES,3892,2226
PT,1885,1788


## Stores (`dim_store`)

**Location:** localities are chosen at random **in proportion to their size** (population in Portugal, number of postal codes in Spain), so big cities are more likely to have a store, as in a real chain. A few stores are placed in mountain regions on purpose, to have snow sales. Each store gets one real postal code of its locality.

In [6]:
def sample_by_size(pool, n):
    # Random choice without repetition, where each locality's chance is proportional to its size
    if len(pool) == 0 or n == 0:
        return pool.iloc[0:0]
    probabilities = pool["SizeIndicator"] / pool["SizeIndicator"].sum()
    positions = rng.choice(len(pool), size=min(n, len(pool)), replace=False, p=probabilities)
    return pool.iloc[positions]


def choose_localities(country, n_stores, n_mountain):
    pool = candidates[candidates["Country"] == country]
    mountain = sample_by_size(pool[pool["IsMountain"]], n_mountain)
    rest = sample_by_size(pool.drop(mountain.index), n_stores - len(mountain))
    return pd.concat([mountain, rest])


chosen = pd.concat([
    choose_localities("PT", N_STORES_PT, N_MOUNTAIN_STORES["PT"]),
    choose_localities("ES", N_STORES_ES, N_MOUNTAIN_STORES["ES"]),
], ignore_index=True)

# One real postal code per chosen locality
store_postal_codes = (
    ref_postal_codes[ref_postal_codes["LocalityID"].isin(chosen["LocalityID"])]
    .groupby("LocalityID")
    .sample(n=1, random_state=SEED)
    [["LocalityID", "PostalCode", "DegreeUrb", "LonCent", "LatCent"]]
)
stores = chosen.merge(store_postal_codes, on="LocalityID")
print(f"{len(stores)} stores")

52 stores


**Store type and size** depend on the degree of urbanisation: shopping centres are more common in cities, retail parks in towns and street stores in rural areas.

| Type | Size (m²) | Opening days |
|---|---|---|
| Shopping centre | 1 200 – 2 000 | Every day, including Sundays and most holidays |
| Street store | 400 – 800 | Closed on Sundays and public holidays |
| Retail park | 2 000 – 3 500 | Every day, including Sundays and most holidays |

In [7]:
type_probabilities = {
    # degree of urbanisation: (shopping_centre, street, retail_park)
    1: [0.6, 0.3, 0.1],   # cities
    2: [0.3, 0.2, 0.5],   # towns and suburbs
    3: [0.0, 0.7, 0.3],   # rural areas
}
size_ranges = {"shopping_centre": (1200, 2000), "street": (400, 800), "retail_park": (2000, 3500)}
urban_labels = {1: "City", 2: "Town", 3: "Rural"}

stores["DegreeUrb"] = stores["DegreeUrb"].fillna(2).astype(int)
stores["store_type"] = [
    rng.choice(["shopping_centre", "street", "retail_park"], p=type_probabilities[degree])
    for degree in stores["DegreeUrb"]
]
stores["size_m2"] = [
    int(round(rng.integers(*size_ranges[store_type]) / 50) * 50)
    for store_type in stores["store_type"]
]
stores["urban_level"] = stores["DegreeUrb"].map(urban_labels)
stores["location_profile"] = np.where(stores["IsMountain"], "Mountain",
                                      np.where(stores["IsCoastal"], "Coastal", "Inland"))

**Opening and closing dates** tell the story of the chain:
* in **Portugal**, about 85% of the stores already existed in 2021 and the others open during the period;
* the **expansion to Spain** starts in 2022, so every Spanish store opens during the period;
* two small street stores in Portugal **close** in 2024.

In [8]:
start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)


def random_dates(n, first, last):
    days = rng.integers(0, (pd.Timestamp(last) - pd.Timestamp(first)).days + 1, size=n)
    return pd.Timestamp(first) + pd.to_timedelta(days, unit="D")


is_pt = stores["Country"] == "PT"
opens_before = is_pt & (rng.random(len(stores)) < 0.85)

stores["opening_date"] = pd.NaT
stores.loc[opens_before, "opening_date"] = random_dates(opens_before.sum(), "2010-01-01", start - pd.Timedelta(days=1))
stores.loc[is_pt & ~opens_before, "opening_date"] = random_dates((is_pt & ~opens_before).sum(), "2021-03-01", "2025-06-30")
stores.loc[~is_pt, "opening_date"] = random_dates((~is_pt).sum(), "2022-03-01", "2025-06-30")

# Two street stores that existed before 2021 close in 2024
stores["closing_date"] = pd.NaT
closing_candidates = stores[opens_before & (stores["store_type"] == "street")].sort_values("size_m2").index[:2]
stores.loc[closing_candidates, "closing_date"] = pd.to_datetime(["2024-06-30", "2024-12-31"][:len(closing_candidates)])

In [9]:
stores = stores.sort_values(["Country", "opening_date"], ascending=[False, True]).reset_index(drop=True)
stores["store_id"] = [f"S{i:03d}" for i in range(1, len(stores) + 1)]
stores["store_name"] = BRAND + " " + stores["LocalityName"]

dim_store = stores.rename(columns={
    "Country": "country", "PostalCode": "postal_code", "LocalityID": "locality_id",
    "PopulationTier": "population_tier", "LonCent": "lon", "LatCent": "lat",
})[[
    "store_id", "store_name", "country", "postal_code", "locality_id", "store_type", "size_m2",
    "opening_date", "closing_date", "location_profile", "urban_level", "population_tier", "lon", "lat",
]]
dim_store.head(10)

,store_id,store_name,country,postal_code,locality_id,store_type,size_m2,opening_date,closing_date,location_profile,urban_level,population_tier,lon,lat
0,S001,Cume & Costa São Bernardo,PT,3810-062,PT_010510,shopping_centre,1650,2010-01-22,NaT,Coastal,City,5,-8.628460,40.617946
1,S002,Cume & Costa Campolide,PT,1070-130,PT_110610,shopping_centre,1550,2010-03-30,NaT,Coastal,City,5,-9.165139,38.727823
2,S003,Cume & Costa Boticas e Granja,PT,5460-324,PT_170219,street,400,2010-05-09,2024-06-30,Mountain,Rural,4,-7.673068,41.691067
3,S004,Cume & Costa Esmoriz,PT,3885-399,PT_011503,shopping_centre,1250,2010-06-29,NaT,Coastal,Town,5,-8.627500,40.957700
4,S005,Cume & Costa Antuzede e Vil de Matos,PT,3025-603,PT_060332,street,600,2010-09-29,NaT,Coastal,City,4,-8.489637,40.276027
5,S006,Cume & Costa Roriz,PT,4795-274,PT_131419,retail_park,3050,2011-01-15,NaT,Coastal,Town,4,-8.365553,41.335673
6,S007,Cume & Costa Santa Clara,PT,1750-192,PT_110664,shopping_centre,1800,2011-05-09,NaT,Coastal,City,5,-9.148281,38.792912
7,S008,Cume & Costa São João de Negrilhos,PT,7600-435,PT_020104,street,550,2011-09-08,NaT,Inland,Rural,3,-8.179457,37.942605
8,S009,Cume & Costa Canidelo,PT,4400-053,PT_131704,street,650,2011-12-22,NaT,Coastal,City,5,-8.644763,41.130005
9,S010,Cume & Costa Cidade da Maia,PT,4470-355,PT_130619,retail_park,3150,2012-02-29,NaT,Coastal,Town,5,-8.605579,41.240485


In [10]:
dim_store.groupby(["country", "store_type"]).size().unstack(fill_value=0)

store_type,retail_park,shopping_centre,street
country,,,
ES,4,3,5
PT,11,13,16


In [11]:
dim_store.groupby(["country", "location_profile"]).size().unstack(fill_value=0)

location_profile,Coastal,Inland,Mountain
country,,,
ES,1,3,8
PT,28,2,10


## Products (`dim_product`)

Six categories with very different seasonality. `base_daily_units` is the average number of units sold per day in a reference store (1 000 m², average location, outside any season or promotion). Expensive products sell less than cheap ones.

Three products tell a story over time: the **e-bike** and the **paddle board** are launched during the period and grow every year, and the **basic fitness tracker** is discontinued when the **GPS sports watch** arrives.

In [12]:
products = [
    # (name, category, base price €, base daily units, launch date, discontinued date)
    ("Road running shoes",       "Running",          89.99, 3.0,  None,         None),
    ("Trail running shoes",      "Running",         109.99, 1.8,  None,         None),
    ("Running T-shirt",          "Running",          19.99, 4.0,  None,         None),
    ("Running shorts",           "Running",          17.99, 3.0,  None,         None),
    ("Running socks (3 pairs)",  "Running",           9.99, 5.0,  None,         None),
    ("GPS sports watch",         "Running",         199.99, 0.25, "2022-09-01", None),
    ("Basic fitness tracker",    "Running",          39.99, 1.2,  None,         "2023-06-30"),
    ("Mountain bike",            "Cycling",         549.00, 0.25, None,         None),
    ("Road bike",                "Cycling",         899.00, 0.12, None,         None),
    ("E-bike",                   "Cycling",        1899.00, 0.02, "2022-03-01", None),
    ("Cycling helmet",           "Cycling",          49.99, 1.0,  None,         None),
    ("Bike lights set",          "Cycling",          24.99, 1.2,  None,         None),
    ("Inner tube",               "Cycling",           6.99, 3.0,  None,         None),
    ("Hiking boots",             "Hiking & Camping", 119.99, 1.2, None,         None),
    ("Backpack 30 L",            "Hiking & Camping",  59.99, 1.0, None,         None),
    ("Tent (2 people)",          "Hiking & Camping", 129.99, 0.3, None,         None),
    ("Sleeping bag",             "Hiking & Camping",  69.99, 0.4, None,         None),
    ("Trekking poles",           "Hiking & Camping",  34.99, 0.6, None,         None),
    ("Headlamp",                 "Hiking & Camping",  19.99, 1.0, None,         None),
    ("Swimsuit",                 "Beach & Water",     29.99, 2.0, None,         None),
    ("Wetsuit",                  "Beach & Water",    149.99, 0.4, None,         None),
    ("Surfboard",                "Beach & Water",    349.00, 0.12, None,        None),
    ("Stand-up paddle board",    "Beach & Water",    399.00, 0.08, "2022-04-01", None),
    ("Snorkel set",              "Beach & Water",     24.99, 0.8, None,         None),
    ("Beach towel",              "Beach & Water",     14.99, 1.5, None,         None),
    ("Ski jacket",               "Snow & Winter",    179.99, 0.3, None,         None),
    ("Ski goggles",              "Snow & Winter",     59.99, 0.3, None,         None),
    ("Thermal base layer",       "Snow & Winter",     34.99, 1.2, None,         None),
    ("Snow gloves",              "Snow & Winter",     29.99, 0.8, None,         None),
    ("Yoga mat",                 "Fitness",           19.99, 1.5, None,         None),
    ("Dumbbell set",             "Fitness",           49.99, 0.6, None,         None),
    ("Resistance bands",         "Fitness",           14.99, 1.5, None,         None),
    ("Training T-shirt",         "Fitness",           14.99, 3.0, None,         None),
]

dim_product = pd.DataFrame(products, columns=[
    "product_name", "category", "base_price", "base_daily_units", "launch_date", "discontinued_date",
])
dim_product.insert(0, "product_id", [f"P{i:03d}" for i in range(1, len(dim_product) + 1)])
dim_product["launch_date"] = pd.to_datetime(dim_product["launch_date"])
dim_product["discontinued_date"] = pd.to_datetime(dim_product["discontinued_date"])

# Unit cost: between 45% and 65% of the price (gross margin between 35% and 55%)
dim_product["unit_cost"] = (dim_product["base_price"] * rng.uniform(0.45, 0.65, len(dim_product))).round(2)

dim_product = dim_product[["product_id", "product_name", "category", "base_price", "unit_cost",
                           "base_daily_units", "launch_date", "discontinued_date"]]
dim_product

,product_id,product_name,category,base_price,unit_cost,base_daily_units,launch_date,discontinued_date
0,P001,Road running shoes,Running,89.99,49.61,3.00,NaT,NaT
1,P002,Trail running shoes,Running,109.99,64.77,1.80,NaT,NaT
2,P003,Running T-shirt,Running,19.99,11.32,4.00,NaT,NaT
3,P004,Running shorts,Running,17.99,8.81,3.00,NaT,NaT
4,P005,Running socks (3 pairs),Running,9.99,6.10,5.00,NaT,NaT
5,P006,GPS sports watch,Running,199.99,118.61,0.25,2022-09-01,NaT
6,P007,Basic fitness tracker,Running,39.99,23.91,1.20,NaT,2023-06-30
7,P008,Mountain bike,Cycling,549.00,261.44,0.25,NaT,NaT
8,P009,Road bike,Cycling,899.00,426.80,0.12,NaT,NaT
9,P010,E-bike,Cycling,1899.00,1206.84,0.02,2022-03-01,NaT


## Calendar (`dim_date`)

One row per day, with the national public holidays of Portugal and Spain (library `holidays`).

In [13]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")
years = list(range(dates.year.min(), dates.year.max() + 1))
holidays_pt = holidays.country_holidays("PT", years=years)
holidays_es = holidays.country_holidays("ES", years=years)

dim_date = pd.DataFrame({"date": dates})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["iso_week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["weekday"] = dim_date["date"].dt.day_of_week + 1      # 1 = Monday ... 7 = Sunday
dim_date["weekday_name"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["weekday"] >= 6
dim_date["season"] = dim_date["month"].map({12: "Winter", 1: "Winter", 2: "Winter",
                                             3: "Spring", 4: "Spring", 5: "Spring",
                                             6: "Summer", 7: "Summer", 8: "Summer",
                                             9: "Autumn", 10: "Autumn", 11: "Autumn"})
dim_date["holiday_pt"] = [holidays_pt.get(d.date()) for d in dim_date["date"]]
dim_date["holiday_es"] = [holidays_es.get(d.date()) for d in dim_date["date"]]
dim_date["is_holiday_pt"] = dim_date["holiday_pt"].notna()
dim_date["is_holiday_es"] = dim_date["holiday_es"].notna()

dim_date[dim_date["is_holiday_pt"] | dim_date["is_holiday_es"]].head(10)

,date,year,quarter,month,month_name,iso_week,weekday,weekday_name,is_weekend,season,holiday_pt,holiday_es,is_holiday_pt,is_holiday_es
0,2021-01-01,2021,1,1,January,53,5,Friday,False,Winter,Ano Novo,Año Nuevo,True,True
5,2021-01-06,2021,1,1,January,1,3,Wednesday,False,Winter,NaN,Epifanía del Señor,False,True
91,2021-04-02,2021,2,4,April,13,5,Friday,False,Spring,Sexta-feira Santa,Viernes Santo,True,True
93,2021-04-04,2021,2,4,April,13,7,Sunday,True,Spring,Páscoa,NaN,True,False
114,2021-04-25,2021,2,4,April,16,7,Sunday,True,Spring,Dia da Liberdade,NaN,True,False
120,2021-05-01,2021,2,5,May,17,6,Saturday,True,Spring,Dia do Trabalhador,Fiesta del Trabajo,True,True
153,2021-06-03,2021,2,6,June,22,4,Thursday,False,Summer,Corpo de Deus,NaN,True,False
160,2021-06-10,2021,2,6,June,23,4,Thursday,False,Summer,"Dia de Portugal, de Camões e das Comunidades P...",NaN,True,False
226,2021-08-15,2021,3,8,August,32,7,Sunday,True,Summer,Assunção de Nossa Senhora,NaN,True,False
277,2021-10-05,2021,4,10,October,40,2,Tuesday,False,Autumn,Implantação da República,NaN,True,False


## Promotions (`dim_promotion`)

The same commercial calendar every year:

| Promotion | Period | Discount | Categories |
|---|---|---|---|
| January sales | 2 Jan – 15 Feb | 25% | Snow & Winter, Hiking & Camping |
| Spring cycling | 1 – 20 Apr | 15% | Cycling |
| Summer sales | 16 Aug – 15 Sep | 30% | Beach & Water, Hiking & Camping |
| Back to sport | 1 – 14 Sep | 15% | Fitness, Running |
| Black Friday | Black Friday to Cyber Monday | 20% | All |

Black Friday is the fourth Friday of November.

In [14]:
def black_friday(year):
    first_november = pd.Timestamp(year=year, month=11, day=1)
    days_to_friday = (4 - first_november.weekday()) % 7      # Friday = 4
    return first_november + pd.Timedelta(days=days_to_friday + 21)


all_categories = ", ".join(sorted(dim_product["category"].unique()))
promotions = []
for year in years:
    bf = black_friday(year)
    promotions += [
        ("January sales", f"{year}-01-02", f"{year}-02-15", 0.25, "Snow & Winter, Hiking & Camping"),
        ("Spring cycling", f"{year}-04-01", f"{year}-04-20", 0.15, "Cycling"),
        ("Summer sales", f"{year}-08-16", f"{year}-09-15", 0.30, "Beach & Water, Hiking & Camping"),
        ("Back to sport", f"{year}-09-01", f"{year}-09-14", 0.15, "Fitness, Running"),
        ("Black Friday", bf, bf + pd.Timedelta(days=3), 0.20, all_categories),
    ]

dim_promotion = pd.DataFrame(promotions, columns=["promotion_name", "start_date", "end_date", "discount_pct", "categories"])
dim_promotion["start_date"] = pd.to_datetime(dim_promotion["start_date"])
dim_promotion["end_date"] = pd.to_datetime(dim_promotion["end_date"])
dim_promotion.insert(0, "promotion_id", [f"PR{i:03d}" for i in range(1, len(dim_promotion) + 1)])
dim_promotion.head(10)

,promotion_id,promotion_name,start_date,end_date,discount_pct,categories
0,PR001,January sales,2021-01-02,2021-02-15,0.25,"Snow & Winter, Hiking & Camping"
1,PR002,Spring cycling,2021-04-01,2021-04-20,0.15,Cycling
2,PR003,Summer sales,2021-08-16,2021-09-15,0.30,"Beach & Water, Hiking & Camping"
3,PR004,Back to sport,2021-09-01,2021-09-14,0.15,"Fitness, Running"
4,PR005,Black Friday,2021-11-26,2021-11-29,0.20,"Beach & Water, Cycling, Fitness, Hiking & Camp..."
5,PR006,January sales,2022-01-02,2022-02-15,0.25,"Snow & Winter, Hiking & Camping"
6,PR007,Spring cycling,2022-04-01,2022-04-20,0.15,Cycling
7,PR008,Summer sales,2022-08-16,2022-09-15,0.30,"Beach & Water, Hiking & Camping"
8,PR009,Back to sport,2022-09-01,2022-09-14,0.15,"Fitness, Running"
9,PR010,Black Friday,2022-11-25,2022-11-28,0.20,"Beach & Water, Cycling, Fitness, Hiking & Camp..."


## Save

In [15]:
save_table(dim_store, "dim_store")
save_table(dim_product, "dim_product")
save_table(dim_date, "dim_date")
save_table(dim_promotion, "dim_promotion")

Saved 'dim_store' (52 rows)
Saved 'dim_product' (33 rows)
Saved 'dim_date' (1,826 rows)
Saved 'dim_promotion' (25 rows)


## Summary
* **Stores:** located at real postal codes, chosen in proportion to the size of the locality, with type, size, location profile (mountain, coastal, inland), opening and closing dates.
* **Products:** 33 products in six categories, with price, cost and base sales, including launches and a discontinued product.
* **Calendar:** one row per day, with the public holidays of Portugal and Spain.
* **Promotions:** five campaigns per year, repeated every year.